# textual smoke test

Smoke test for the `textual` TUI framework. Exercises imports + a few primary APIs without spinning up an actual terminal app loop.

In [ ]:
# Example 1: define an App subclass and inspect its composition without running it.
from textual.app import App
from textual.widgets import Static, Header, Footer

class HelloApp(App):
    TITLE = "Hello"
    def compose(self):
        yield Header()
        yield Static("Hello, Textual!")
        yield Footer()

app = HelloApp()
print(f"App class: {type(app).__name__}")
print(f"App title: {app.TITLE}")
print(f"App is_running: {app.is_running}")

# Walk the compose generator without mounting widgets (which requires an active app loop).
composed = list(app.compose())
print(f"Composed widget count: {len(composed)}")
print(f"Composed widget types: {[type(w).__name__ for w in composed]}")

assert len(composed) == 3
assert isinstance(composed[1], Static)

In [ ]:
# Example 2: reactive attributes on a Widget subclass.
from textual.reactive import reactive
from textual.widget import Widget

class Counter(Widget):
    count = reactive(0)
    label = reactive("start")

# Inspect the reactive descriptors on the class.
print(f"Counter reactive descriptors: {list(Counter._reactives.keys())}")
print(f"count default: {Counter._reactives['count']._default}")
print(f"label default: {Counter._reactives['label']._default}")

assert 'count' in Counter._reactives
assert 'label' in Counter._reactives

In [ ]:
# Example 3: Color parsing and manipulation.
from textual.color import Color

red = Color.parse("red")
print(f"red = {red!r}")
print(f"red.rgb = {red.rgb}")
print(f"red.hex = {red.hex}")

blue = Color.parse("#0000ff")
print(f"blue = {blue!r}")

# Blend two colors.
mix = red.blend(blue, factor=0.5)
print(f"red.blend(blue, 0.5) = {mix!r}")

darker = red.darken(0.3)
print(f"red.darken(0.3) = {darker!r}")

assert red.rgb == (255, 0, 0)
assert blue.rgb == (0, 0, 255)

In [ ]:
# Example 4: Binding objects (keyboard bindings declared on Apps/Widgets).
from textual.binding import Binding

b1 = Binding(key="q", action="quit", description="Quit")
b2 = Binding(key="ctrl+c", action="quit", description="Quit (ctrl-c)", show=False)

print(f"b1 = {b1}")
print(f"b1.key = {b1.key}, action = {b1.action}, description = {b1.description}")
print(f"b2.show = {b2.show}")

# Bindings declared as a class attribute on an App.
from textual.app import App
class BoundApp(App):
    BINDINGS = [
        Binding("q", "quit", "Quit"),
        ("r", "reload", "Reload"),
    ]

print(f"BoundApp.BINDINGS count: {len(BoundApp.BINDINGS)}")
assert b1.key == "q"
assert b1.action == "quit"
assert len(BoundApp.BINDINGS) == 2

In [ ]:
# Example 5: Message classes — the core inter-widget event system.
from textual.message import Message

class Ping(Message):
    def __init__(self, payload: str) -> None:
        self.payload = payload
        super().__init__()

msg = Ping("hello")
print(f"msg = {msg!r}")
print(f"msg.payload = {msg.payload}")
print(f"msg.handler_name = {msg.handler_name}")
print(f"msg.bubble = {msg.bubble}")

# handler_name is auto-derived from the class name (Ping -> on_ping).
assert msg.handler_name == "on_ping"
assert msg.payload == "hello"
print("\ntextual smoke test PASSED")